# 05 - Model Training

## Objective

This notebook trains and compares multiple machine learning models for detecting suspicious financial transactions.

Models to compare:

- Dummy Classifier (Baseline)
- Logistic Regression
- Decision Tree
- Random Forest

The best model will be selected using cross-validation and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

import joblib

In [ ]:
df = pd.read_csv("../data/processed/engineered_transactions.csv")

df.head()

In [ ]:
if "transaction_id" in df.columns:
    df.drop("transaction_id", axis=1, inplace=True)

In [ ]:
X = df.drop("is_fraud", axis=1)

y = df["is_fraud"]

In [ ]:
numeric_features = X.select_dtypes(include=["int64","float64"]).columns

categorical_features = X.select_dtypes(include=["object","category"]).columns

print(numeric_features)

print(categorical_features)

In [ ]:
preprocessor = ColumnTransformer(

    transformers=[

        ("num", StandardScaler(), numeric_features),

        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)

    ]

)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    stratify=y,

    random_state=42

)

print(X_train.shape)

print(X_test.shape)

In [ ]:
baseline = Pipeline([

    ("preprocessor",preprocessor),

    ("classifier",DummyClassifier(strategy="most_frequent"))

])

baseline.fit(X_train,y_train)

pred = baseline.predict(X_test)

print("Baseline Accuracy:",accuracy_score(y_test,pred))

In [ ]:
lr_pipeline = Pipeline([

    ("preprocessor",preprocessor),

    ("classifier",LogisticRegression(max_iter=1000))

])

lr_pipeline.fit(X_train,y_train)

In [ ]:
dt_pipeline = Pipeline([

    ("preprocessor",preprocessor),

    ("classifier",DecisionTreeClassifier(random_state=42))

])

dt_pipeline.fit(X_train,y_train)

In [ ]:
rf_pipeline = Pipeline([

    ("preprocessor",preprocessor),

    ("classifier",RandomForestClassifier(random_state=42))

])

rf_pipeline.fit(X_train,y_train)

In [ ]:
models = {

    "Logistic Regression":lr_pipeline,

    "Decision Tree":dt_pipeline,

    "Random Forest":rf_pipeline

}

for name,model in models.items():

    scores = cross_val_score(

        model,

        X_train,

        y_train,

        cv=5,

        scoring="f1"

    )

    print(name)

    print("Average F1:",scores.mean())

    print("--------------------")

In [ ]:
parameters = {

    "classifier__n_estimators":[100,200],

    "classifier__max_depth":[5,10,None]

}

grid = GridSearchCV(

    rf_pipeline,

    parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

grid.fit(X_train,y_train)

print("Best Parameters")

print(grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

prediction = best_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test,prediction)

precision = precision_score(y_test,prediction)

recall = recall_score(y_test,prediction)

f1 = f1_score(y_test,prediction)

roc = roc_auc_score(y_test,prediction)

print("Accuracy :",accuracy)

print("Precision:",precision)

print("Recall   :",recall)

print("F1 Score :",f1)

print("ROC AUC  :",roc)

In [ ]:
results = pd.DataFrame({

    "Metric":[

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score",

        "ROC AUC"

    ],

    "Score":[

        accuracy,

        precision,

        recall,

        f1,

        roc

    ]

})

results

In [ ]:
joblib.dump(

    best_model,

    "../models/fraud_model.pkl"

)

print("Model Saved Successfully")

In [ ]:
with open("../models/model_metrics.txt","w") as file:

    file.write(f"Accuracy : {accuracy}\n")

    file.write(f"Precision: {precision}\n")

    file.write(f"Recall   : {recall}\n")

    file.write(f"F1 Score : {f1}\n")

    file.write(f"ROC AUC  : {roc}\n")

print("Metrics Saved Successfully")

# Model Selection

Models Compared

- Dummy Classifier
- Logistic Regression
- Decision Tree
- Random Forest

Hyperparameter tuning was performed on the Random Forest model using GridSearchCV.

The best-performing model was selected based on the F1 Score.

The trained model has been saved as:

models/fraud_model.pkl